In [1]:
import pulp
import pandas as pd
from itertools import combinations
from enum import Enum

In [2]:
def gerar_grade_horaria_por_turma(tabela, instancia, turma_escolhida):
    horarios = instancia["horarios"]
    horarios_legenda = instancia["horarios_legenda"]
    dias = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta"]

    codigos_horario = sorted(
        {h.split("_")[1] for h in horarios},
        key=lambda x: (x[0], int(x[1]))
    )
    eixo_y = [horarios_legenda[codigo] for codigo in codigos_horario]
    grade = pd.DataFrame("", index=eixo_y, columns=dias)
    aulas_turma = tabela[tabela["turma"] == turma_escolhida]

    for _, aula in aulas_turma.iterrows():
        dia, codigo = aula["horario"].split("_")
        horario_legivel = horarios_legenda[codigo]

        conteudo = (
            f"{aula['disciplina']} - {aula['professor']} - {aula['sala']}"
        )

        grade.loc[horario_legivel, dia] = conteudo

    return grade

def exportar_grades_para_excel(tabela, instancia, caminho_arquivo_excel="grade_horaria_escola.xlsx", pasta=None):
    turmas = instancia["turmas"]

    with pd.ExcelWriter(f"{pasta}/{caminho_arquivo_excel}", engine="openpyxl") as writer:
        for turma in turmas:
            grade = gerar_grade_horaria_por_turma(tabela, instancia, turma)
            nome_aba = f"Turma_{turma}"[:31]
            grade.to_excel(writer, sheet_name=nome_aba)

    print(f"Arquivo Excel gerado: {caminho_arquivo_excel}")


In [3]:
class Disciplina(str, Enum):
    Matematica = "Matemática"
    Fisica = "Física"
    Quimica = "Química"
    LinguaPortuguesa = "Lingua Portuguesa"
    Lingua = "Lingua Inglesa"
    Filosofia = "Filosofia"
    Sociologia = "Sociologia"
    Geografia = "Geografia"
    História = "História"
    EducacaoFisica = "Educação Fisica"
    Biologia = "Biologia"
    Arte = "Arte"
    Inovativa = "Inovativa"
    Senac = "Senac"
    CESAR = "CESAR"
    CulturaDigital = "Cultura digital"
    EstudoOrientado = "Estudo orientado"

print(Disciplina.Matematica.value)

Matemática


In [4]:
def montar_instancia(turmas_por_3_ano : int, turmas_por_2_ano : int, total_salas : int):
    materias_regulares = ["Matemática", "Física", "Química", "Lingua portuguesa", "Lingua Inglesa", "Filosofia", "Sociologia", "Geografia", "História", "Educação Fisica", "Biologia", "Arte"]
    programas_formativos = ["Senac", "CESAR", "Inovativa"]
    disciplinas_complementares = ["Cultura digital", "Estudo orientado"]
    professores = [
                    "prof1", "prof2", "prof3", "prof4", "prof5", "prof6", "prof7", "prof8", "prof9", "prof10", "prof11", "prof12", 
                    "prof_int1",
                    "prof_ext1", "prof_ext2",
                    "prof_comp1", "prof_comp2"
                    ]

    horarios = ["Segunda_M1", "Segunda_M2", "Segunda_M3", "Segunda_M4", "Segunda_M5", "Segunda_T1", "Segunda_T2", "Segunda_T3", "Segunda_T4", 
                "Terça_M1", "Terça_M2", "Terça_M3", "Terça_M4", "Terça_M5", "Terça_T1", "Terça_T2", "Terça_T3", "Terça_T4",
                "Quarta_M1", "Quarta_M2", "Quarta_M3", "Quarta_M4", "Quarta_M5", "Quarta_T1", "Quarta_T2", "Quarta_T3", "Quarta_T4",
                "Quinta_M1", "Quinta_M2", "Quinta_M3", "Quinta_M4", "Quinta_M5", "Quinta_T1", "Quinta_T2", "Quinta_T3", "Quinta_T4",
                "Sexta_M1", "Sexta_M2", "Sexta_M3", "Sexta_M4", "Sexta_M5", "Sexta_T1", "Sexta_T2", "Sexta_T3", "Sexta_T4"
                ]
    horarios_legenda = {"M1": "7:30 às 8:20", "M2": "8:20 às 9:10", "M3": "9:30 às 10:20", "M4": "10:20 às 11:10", "M5": "11:10 às 12:00", 
                        "T1": "13:20 às 14:10", "T2": "14:10 às 15:00", "T3": "15:20 às 16:10", "T4": "16:10 às 17:00",}

    professor_pode_lecionar = {
        ("prof1", "Matemática"): 1,
        ("prof2", "Física"): 1,
        ("prof3", "Química"): 1,
        ("prof4", "Lingua portuguesa"): 1,
        ("prof5", "Lingua Inglesa"): 1,
        ("prof6", "Filosofia"): 1,
        ("prof7", "Sociologia"): 1,
        ("prof8", "Geografia"): 1,
        ("prof9", "História"): 1,
        ("prof10", "Educação Fisica"): 1,
        ("prof11", "Biologia"): 1,
        ("prof12", "Arte"): 1,
        ("prof_int1", "Inovativa"): 1,
        ("prof_ext1", "Senac"): 1,
        ("prof_ext2", "CESAR"): 1,
        ("prof_comp1", "Cultura digital"): 1,
        ("prof_comp2", "Estudo orientado"): 1,
    }

    disponibilidade = {}
        
    for professor in professores:
        for horario in horarios:

            if professor.startswith("prof") and professor[4:].isdigit():
                disponibilidade[(professor, horario)] = 1

            elif professor.startswith("prof_int") or professor.startswith("prof_ext"):
                if "_T" in horario:
                    disponibilidade[(professor, horario)] = 1
                else:
                    disponibilidade[(professor, horario)] = 0

            elif professor.startswith("prof_comp"):
                if "_T" in horario:
                    disponibilidade[(professor, horario)] = 1
                else:
                    disponibilidade[(professor, horario)] = 0

    salas = []
    for s in range(1, total_salas+1):
        salas += [f"Sala_{s}"]

    turmas = []
    ano_turma = {}
    turma_sala = {}

    turmas_por_1_ano = total_salas - turmas_por_2_ano - turmas_por_3_ano
    turma_sala_pos = 0
    for numero_ano_turma, quant_turma in enumerate([turmas_por_1_ano, turmas_por_2_ano, turmas_por_3_ano]):
        for t in range(quant_turma):
            cod_turma = f"{numero_ano_turma+1}{chr(65+t)}"
            turmas += [cod_turma]
            ano_turma[cod_turma] = numero_ano_turma+1
            turma_sala[cod_turma] = salas[turma_sala_pos]
            turma_sala_pos += 1

    quantidade_aulas = {}
    for turma in turmas:
        quantidade_aulas[(turma, "Arte")] = 1
        quantidade_aulas[(turma, "Biologia")] = 2
        quantidade_aulas[(turma, "Educação Fisica")] = 1
        quantidade_aulas[(turma, "Filosofia")] = 1
        quantidade_aulas[(turma, "Física")] = 2
        quantidade_aulas[(turma, "Geografia")] = 2
        quantidade_aulas[(turma, "História")] = 2
        quantidade_aulas[(turma, "Lingua Inglesa")] = 2
        quantidade_aulas[(turma, "Lingua portuguesa")] = 4
        quantidade_aulas[(turma, "Matemática")] = 4
        quantidade_aulas[(turma, "Química")] = 2
        quantidade_aulas[(turma, "Sociologia")] = 1

        if (ano_turma[turma] == 3):
            quantidade_aulas[(turma, "Matemática")] = 5
            quantidade_aulas[(turma, "Lingua portuguesa")] = 5

    return {
        "turmas": turmas,
        "ano_da_turma": ano_turma,
        "materias_regulares": materias_regulares,
        "professores": professores,
        "horarios": horarios,
        "horarios_legenda": horarios_legenda,
        "salas": salas,
        "professor_pode_lecionar": professor_pode_lecionar,
        "professor_disponivel_no_horario": disponibilidade,
        "quantidade_aulas": quantidade_aulas,
        "disciplinas_complementares": disciplinas_complementares,
        "programas_formativos": programas_formativos,
        "turma_sala": turma_sala
    }


In [5]:
def construir_e_resolver_modelo(instancia, limite_tempo=120):
    # Variaveis
    turmas = instancia["turmas"]
    ano_da_turma = instancia["ano_da_turma"]
    turma_sala = instancia["turma_sala"]
    professores = instancia["professores"]
    horarios = instancia["horarios"]

    materias_regulares = instancia["materias_regulares"]
    disciplinas_complementares = set(instancia.get("disciplinas_complementares", []))
    programas_formativos = set(instancia.get("programas_formativos", []))
    disciplinas = (
        set(materias_regulares)
        | set(programas_formativos)
        | set(disciplinas_complementares)
    )

    # Algumas limitacoes
    professor_pode_lecionar = instancia["professor_pode_lecionar"]
    professor_disponivel_no_horario = instancia["professor_disponivel_no_horario"]
    quantidade_aulas = instancia["quantidade_aulas"]

    atribuicoes_fixas = set(instancia.get("atribuicoes_fixas", []))

    def contar_fixas(turma, conjunto_disciplinas):
        return sum(
            1 for (t, d, _, _) in atribuicoes_fixas
            if t == turma and d in conjunto_disciplinas
        )

    #  VARIÁVEIS DE DECISÃO
    #  x[turma, disciplina, professor, horario] = 1 se a aula ocorre
    variavel_alocacao = {}
    problema = pulp.LpProblem("Alocacao_Aulas_Escolar", pulp.const.LpMaximize)

    for turma in turmas:
        for disciplina in disciplinas:
            for professor in professores:
                for horario in horarios:

                    if professor_pode_lecionar.get((professor, disciplina), 0) == 0:
                        continue

                    if professor_disponivel_no_horario.get((professor, horario), 0) == 0:
                        continue

                    variavel_alocacao[(turma, disciplina, professor, horario)] = (
                        pulp.LpVariable(
                            f"alocar_{turma}_{disciplina}_{professor}_{horario}",
                            cat="Binary"
                        )
                    )

    for (turma, disciplina, professor, horario) in atribuicoes_fixas:
        chave = (turma, disciplina, professor, horario)
        if chave not in variavel_alocacao:
            raise ValueError(f"Atribuição fixa inválida: {chave}")

        problema += (
            variavel_alocacao[chave] == 1,
            f"fixa_{turma}_{disciplina}_{professor}_{horario}"
        )

    # Objetivo: maximizar a alocação de aulas obrigatórias no turno da manha
    horarios_manha = [h for h in horarios if "_M" in h]

    problema += pulp.lpSum(
        variavel_alocacao.get((turma, disciplina, professor, horario), 0)
        for turma in turmas
        for disciplina in materias_regulares
        for professor in professores
        for horario in horarios_manha
    ), "Maximizar_Obrigatorias_Manha"

    #  RESTRIÇÃO 1 - ATE UMA AULA POR TURMA POR HORÁRIO 
    for turma in turmas:
        for horario in horarios:

            problema += (
                pulp.lpSum(
                    variavel_alocacao.get((turma, disciplina, professor, horario), 0)
                    for disciplina in disciplinas
                    for professor in professores
                ) <= 1,
                f"uma_aula_por_turma_por_horario_{turma}_{horario}"
            )

    #  RESTRIÇÃO 2 - UM PROFESSOR POR HORÁRIO
    for professor in professores:
        for horario in horarios:

            problema += (
                pulp.lpSum(
                    variavel_alocacao.get((turma, disciplina, professor, horario), 0)
                    for turma in turmas
                    for disciplina in disciplinas
                ) <= 1,
                f"um_professor_por_horario_{professor}_{horario}"
            )

    #  RESTRIÇÃO 3 - QUANTIDADE OBRIGATORIA DE AULAS DE DISCIPLINAS OBRIGATORIAS 
    for turma in turmas:
        for disciplina in materias_regulares:

            problema += (
                pulp.lpSum(
                    variavel_alocacao.get((turma, disciplina, professor, horario), 0)
                    for professor in professores
                    for horario in horarios
                ) == quantidade_aulas[(turma, disciplina)],
                f"cobertura_{turma}_{disciplina}"
            )

    # RESTRIÇÕES 4 — REGRAS POR ANO DA TURMA

    # 1 ANO - 
    turmas_ano1 = [t for t in turmas if ano_da_turma[t] == 1]
    itinerarios_usados = {
        d for (t, d, _, _) in atribuicoes_fixas
        if t in turmas_ano1 and d in programas_formativos
    }
    turmas_ano1_com_fixa = {
        t for (t, d, _, _) in atribuicoes_fixas
        if t in turmas_ano1 and d in programas_formativos
    }
    turmas_sem_fixa = set(turmas_ano1) - turmas_ano1_com_fixa

    # Turmas COM itinerário fixo proíbe qualquer outro itinerário
    for turma in turmas_ano1_com_fixa:
        disciplina_fixa = next(
            d for (t, d, _, _) in atribuicoes_fixas
            if t == turma and d in programas_formativos
        )

        for d in programas_formativos:
            if d != disciplina_fixa:
                for professor in professores:
                    for horario in horarios:
                        chave = (turma, d, professor, horario)
                        if chave in variavel_alocacao:
                            problema += (
                                variavel_alocacao[chave] == 0,
                                f"bloqueio_itinerario_{turma}_{d}_{horario}"
                            )

    # Turmas SEM itinerário fixo só podem usar itinerários ainda não usados
    for turma in turmas_sem_fixa:
        for d in itinerarios_usados:
            for professor in professores:
                for horario in horarios:
                    chave = (turma, d, professor, horario)
                    if chave in variavel_alocacao:
                        problema += (
                            variavel_alocacao[chave] == 0,
                            f"bloqueio_itinerario_usado_{turma}_{d}_{horario}"
                        )

    # Turmas do 1º ano exatamente 2 aulas de UM único itinerário
    for turma in turmas_ano1:
        problema += (
            pulp.lpSum(
                variavel_alocacao.get((turma, d, professor, horario), 0)
                for d in programas_formativos
                for professor in professores
                for horario in horarios
            ) == 2,
            f"ano1_exatamente_duas_aulas_itinerario_{turma}"
        )

    # 2 ANO - 
    turmas_ano2 = [t for t in turmas if ano_da_turma[t] == 2]
    itinerarios_usados_ano2 = {
        d for (t, d, _, _) in atribuicoes_fixas
        if t in turmas_ano2 and d in programas_formativos
    }
    turmas_ano2_com_fixa = {
        t for (t, d, _, _) in atribuicoes_fixas
        if t in turmas_ano2 and d in programas_formativos
    }
    turmas_ano2_sem_fixa = set(turmas_ano2) - turmas_ano2_com_fixa

    # Turmas COM itinerário fixo. Proíbe qualquer outro itinerário
    for turma in turmas_ano2_com_fixa:
        disciplina_fixa = next(
            d for (t, d, _, _) in atribuicoes_fixas
            if t == turma and d in programas_formativos
        )

        for d in programas_formativos:
            if d != disciplina_fixa:
                for professor in professores:
                    for horario in horarios:
                        chave = (turma, d, professor, horario)
                        if chave in variavel_alocacao:
                            problema += (
                                variavel_alocacao[chave] == 0,
                                f"ano2_bloqueio_itinerario_{turma}_{d}_{horario}"
                            )

        # exatamente 2 aulas do itinerário fixo
        problema += (
            pulp.lpSum(
                variavel_alocacao.get((turma, disciplina_fixa, professor, horario), 0)
                for professor in professores
                for horario in horarios
            ) == 2,
            f"ano2_duas_aulas_itinerario_fixo_{turma}"
        )

    # Turmas SEM itinerário fixo. Itinerários não usados OU complementares
    for turma in turmas_ano2_sem_fixa:

        # Bloqueia itinerários já usados
        for d in itinerarios_usados_ano2:
            for professor in professores:
                for horario in horarios:
                    chave = (turma, d, professor, horario)
                    if chave in variavel_alocacao:
                        problema += (
                            variavel_alocacao[chave] == 0,
                            f"ano2_bloqueio_itinerario_usado_{turma}_{d}_{horario}"
                        )

        # Total de aulas (itinerário + complementares) = 2
        problema += (
            pulp.lpSum(
                variavel_alocacao.get((turma, d, p, h), 0)
                for d in (programas_formativos | disciplinas_complementares)
                for p in professores
                for h in horarios
            ) == 2,
            f"ano2_total_duas_aulas_{turma}"
        )

        # No máximo 1 itinerário (se escolher itinerário, as 2 aulas são dele)
        # Não permitir misturar itinerários
        for d1, d2 in combinations(programas_formativos, 2):
            problema += (
                pulp.lpSum(
                    variavel_alocacao.get((turma, d1, p, h), 0)
                    for p in professores
                    for h in horarios
                )
                +
                pulp.lpSum(
                    variavel_alocacao.get((turma, d2, p, h), 0)
                    for p in professores
                    for h in horarios
                ) <= 2,
                f"ano2_nao_mistura_{turma}_{d1}_{d2}"
            )

        # Cada disciplina complementar no máximo 1 vez
        for d in disciplinas_complementares:
            problema += (
                pulp.lpSum(
                    variavel_alocacao.get((turma, d, p, h), 0)
                    for p in professores
                    for h in horarios
                ) <= 1,
                f"ano2_complementar_unica_{turma}_{d}"
            )

    # 3 ANO - exatamente 1 aula de cada disciplina complementar
    for turma in turmas:
        if ano_da_turma[turma] == 3:
            for disciplina in disciplinas_complementares:
                problema += (
                    pulp.lpSum(
                        variavel_alocacao.get((turma, disciplina, professor, horario), 0)
                        for professor in professores
                        for horario in horarios
                    ) == 1,
                    f"ano3_complementar_{turma}_{disciplina}"
                )

    #  SOLVER
    solver = pulp.PULP_CBC_CMD(msg=True, timeLimit=limite_tempo, threads=2)
    status = problema.solve(solver)

    print("\nSTATUS DO SOLVER:", pulp.LpStatus[problema.status])

    atribuicoes = []

    # Atribuições fixas
    for (turma, disc, professor, horario) in atribuicoes_fixas:
        atribuicoes.append({
            "turma": turma,
            "disciplina": disc,
            "professor": professor,
            "horario": horario,
            "sala": turma_sala[turma],
            "fixa": True
        })

    # Atribuições decididas pelo modelo
    for chave, var in variavel_alocacao.items():
        if pulp.value(var) is not None and pulp.value(var) == 1:
            turma, disc, professor, horario = chave
            atribuicoes.append({
                "turma": turma,
                "disciplina": disc,
                "professor": professor,
                "horario": horario,
                "sala": turma_sala[turma],
                "fixa": False
            })

    df = pd.DataFrame(atribuicoes)
    if not df.empty:
        df = df.sort_values(["horario", "turma", "disciplina"]).reset_index(drop=True)

    return problema, df, variavel_alocacao, pulp

In [6]:
# x[turma, disciplina, professor, horario] = 1
atribuicoes_fixas = {
    ("1A", "Senac", "prof_ext1", "Quarta_T1"),
    ("1A", "Senac", "prof_ext1", "Quinta_T1"),
    ("1B", "CESAR", "prof_ext2", "Segunda_T1"),
    ("1B", "CESAR", "prof_ext2", "Quarta_T1"),
    ("2A", "CESAR", "prof_ext2", "Terça_T1"),
    ("2A", "CESAR", "prof_ext2", "Quinta_T1"),
}
instancia = montar_instancia(turmas_por_3_ano=3, turmas_por_2_ano=3, total_salas=9)
instancia["atribuicoes_fixas"] = atribuicoes_fixas
instancia

{'turmas': ['1A', '1B', '1C', '2A', '2B', '2C', '3A', '3B', '3C'],
 'ano_da_turma': {'1A': 1,
  '1B': 1,
  '1C': 1,
  '2A': 2,
  '2B': 2,
  '2C': 2,
  '3A': 3,
  '3B': 3,
  '3C': 3},
 'materias_regulares': ['Matemática',
  'Física',
  'Química',
  'Lingua portuguesa',
  'Lingua Inglesa',
  'Filosofia',
  'Sociologia',
  'Geografia',
  'História',
  'Educação Fisica',
  'Biologia',
  'Arte'],
 'professores': ['prof1',
  'prof2',
  'prof3',
  'prof4',
  'prof5',
  'prof6',
  'prof7',
  'prof8',
  'prof9',
  'prof10',
  'prof11',
  'prof12',
  'prof_int1',
  'prof_ext1',
  'prof_ext2',
  'prof_comp1',
  'prof_comp2'],
 'horarios': ['Segunda_M1',
  'Segunda_M2',
  'Segunda_M3',
  'Segunda_M4',
  'Segunda_M5',
  'Segunda_T1',
  'Segunda_T2',
  'Segunda_T3',
  'Segunda_T4',
  'Terça_M1',
  'Terça_M2',
  'Terça_M3',
  'Terça_M4',
  'Terça_M5',
  'Terça_T1',
  'Terça_T2',
  'Terça_T3',
  'Terça_T4',
  'Quarta_M1',
  'Quarta_M2',
  'Quarta_M3',
  'Quarta_M4',
  'Quarta_M5',
  'Quarta_T1',
  'Qu

In [7]:
solucao, tabela, variavel_alocacao, pulp = construir_e_resolver_modelo(instancia)

print("Objetivos:", pulp.value(solucao.objective))
print("Restrições:", len(solucao.constraints))
print("Variáveis:", len(solucao.variables()))

if tabela.empty:
    print("Nenhuma atribuição encontrada — instância possivelmente inviável ou precisa de tempo maior.")
else:
    print("\nGrade encontrada:")
    print(tabela.to_string())



STATUS DO SOLVER: Optimal
Objetivos: 194.0
Restrições: 1506
Variáveis: 5760

Grade encontrada:
    turma         disciplina   professor     horario    sala   fixa
0      1A            Química       prof3   Quarta_M1  Sala_1  False
1      1C  Lingua portuguesa       prof4   Quarta_M1  Sala_3  False
2      2A           História       prof9   Quarta_M1  Sala_4  False
3      2B     Lingua Inglesa       prof5   Quarta_M1  Sala_5  False
4      2C          Geografia       prof8   Quarta_M1  Sala_6  False
5      3A           Biologia      prof11   Quarta_M1  Sala_7  False
6      3B             Física       prof2   Quarta_M1  Sala_8  False
7      3C         Matemática       prof1   Quarta_M1  Sala_9  False
8      1A            Química       prof3   Quarta_M2  Sala_1  False
9      1B    Educação Fisica      prof10   Quarta_M2  Sala_2  False
10     1C          Geografia       prof8   Quarta_M2  Sala_3  False
11     2A     Lingua Inglesa       prof5   Quarta_M2  Sala_4  False
12     2B         Ma

In [11]:
gerar_grade_horaria_por_turma(tabela, instancia, instancia["turmas"][2])

,Segunda,Terça,Quarta,Quinta,Sexta
7:30 às 8:20,História - prof9 - Sala_3,Lingua portuguesa - prof4 - Sala_3,Lingua portuguesa - prof4 - Sala_3,Sociologia - prof7 - Sala_3,Física - prof2 - Sala_3
8:20 às 9:10,Física - prof2 - Sala_3,Educação Fisica - prof10 - Sala_3,Geografia - prof8 - Sala_3,Lingua Inglesa - prof5 - Sala_3,Matemática - prof1 - Sala_3
9:30 às 10:20,Matemática - prof1 - Sala_3,Filosofia - prof6 - Sala_3,Geografia - prof8 - Sala_3,Matemática - prof1 - Sala_3,Química - prof3 - Sala_3
10:20 às 11:10,Lingua portuguesa - prof4 - Sala_3,Biologia - prof11 - Sala_3,História - prof9 - Sala_3,Matemática - prof1 - Sala_3,Lingua Inglesa - prof5 - Sala_3
11:10 às 12:00,,Arte - prof12 - Sala_3,Química - prof3 - Sala_3,Lingua portuguesa - prof4 - Sala_3,Biologia - prof11 - Sala_3
13:20 às 14:10,Inovativa - prof_int1 - Sala_3,,Inovativa - prof_int1 - Sala_3,,
14:10 às 15:00,,,,,
15:20 às 16:10,,,,,
16:10 às 17:00,,,,,


In [9]:
exportar_grades_para_excel(tabela, instancia, caminho_arquivo_excel="grade_horaria_padre_machado.xlsx", pasta="rascunhos")

Arquivo Excel gerado: grade_horaria_padre_machado.xlsx
